# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MusaGaya/KGaya/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Lane 2: Refresh / Content Opportunity Scoring

Task type: Ranking / Scoring

This is a scoring problem. The goal is not to classify pages into a hard
"yes/no" bucket, but to assign each page a score that reflects how urgently
it needs a content review. The output is a ranked list — pages with the
highest scores get reviewed first. This maps most naturally to a ranking
or scoring task, though the underlying model uses binary classification
(declining vs not declining) to generate the probabilities that drive the score.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target / Proxy: is_declining_label

The target column is is_declining_label, derived from trend_direction == "down".
This is a proxy label — it captures pages that are currently showing a declining
trend in the available 90-day window, not a future-looking outcome. It is used
as a stand-in for the real question: which pages will lose visibility if not
reviewed soon?

This proxy has a known weakness: it is calculated from the same window as the
features, which means it is not truly predicting the future. A stronger label
would compare a prior feature window against a future outcome window. For now,
this proxy is sufficient to demonstrate the framing and pipeline.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Success metric: Precision@K

The right metric for this lane is Precision@K — specifically Precision@50.
This asks: of the top 50 pages the model flags for review, how many are
actually declining?

This metric matches how the output is actually used: a content reviewer
has limited time and will only act on a small number of pages. Getting the
top of the list right matters far more than overall accuracy across all 30,000 rows.

From the starter pipeline results:
- Baseline rule Precision@50: 0.240 (about 12 of the top 50 correct)
- Random forest Precision@50: 0.740 (about 37 of the top 50 correct)

The target for this lane is to match or exceed the starter random forest
result of 0.740 on Precision@50 using the warehouse data.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

The unit of analysis is one content item — one page. Each row in the
dataframe represents a single page with its 90-day performance metrics.
The model scores each page independently and the final output is a ranked
list of pages ordered by their review priority score.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess
import pandas as pd
import numpy as np

IN_COLAB = "google.colab" in sys.modules
REPO_DIR = "flyrank-ml-internship-starter"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/flyrank-bih/flyrank-ml-internship-starter",
                        REPO_DIR], check=True)
    os.chdir(REPO_DIR)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# One row = one content item (one page)
print(f"Shape: {df.shape}")
print(f"One row = one content item (page)")
print(f"\nUnit of analysis preview:")
df[['content_id', 'impressions_90d', 'content_age_days',
    'avg_position', 'ctr', 'trend_direction']].head(10)

Shape: (30000, 44)
One row = one content item (page)

Unit of analysis preview:


,content_id,impressions_90d,content_age_days,avg_position,ctr,trend_direction
0,content_304f48230142,3803,187,10.6,0.76,down
1,content_a1fb4e703a9e,15320,445,20.3,0.05,down
2,content_9aa793d4d895,12581,141,36.5,0.09,down
3,content_331d6c4de07b,11751,463,6.2,0.49,stable
4,content_d99b7a2d90ca,19140,263,44.0,0.13,down
5,content_d4084a4bc775,3970,147,8.5,0.03,down
6,content_9a34b442b552,20,90,7.0,0.00,down
7,content_a63219c6e95a,1724,445,21.2,0.06,stable
8,content_5e6c160719bc,32574,90,46.0,0.09,down
9,content_c27558df2b0c,1240,257,4.9,0.16,down


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule like "flag pages that are stale and still visible" can only
combine two or three signals using hand-tuned thresholds. It treats all
stale visible pages the same regardless of whether they are actually declining.

ML improves on this because:

1. It combines many signals at once — impressions, position, CTR, content age,
   word count, freshness — and learns which combinations actually predict decline.

2. It finds non-obvious patterns — for example, the starter decision tree
   discovered that pages with more than 5.5 impressions AND content younger
   than 312 days are more likely to be declining. A human writing rules would
   not have guessed that specific threshold.

3. It ranks continuously — instead of a binary flag, it produces a probability
   score for every page, which means the reviewer always gets a properly ordered
   list rather than a flat group of flagged pages.

The starter pipeline already demonstrated this directionally: the random forest
achieved Precision@50 of 0.740 versus the hand rule's 0.240 — roughly 3x better
at identifying the right pages to review first.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.